# Selected email/phone masking

Use this folder and the shared root kernel. All inputs are synthetic. The baseline recognizes conventional email and dashed US-style phone strings; it is not an anonymizer.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/inputs.json").read_text())
pprint(fixtures)

## Read the small detector first

`tools/redact.py` contains `detect_entities` (matched spans), `mask_text` (replacement), and `guarded_request` (input mask before responder/telemetry, then output mask). The first two functions return ordinary decisions and reasons.

In [ ]:
from tools.redact import detect_entities, mask_text, guarded_request
for item in fixtures:
    pprint({"id": item["id"], "original": item["text"], "check": mask_text(item["text"])})

## Inspect both boundaries

`models/demo.py` is a local function, not a model. It echoes the actual text received and appends a synthetic email. Our local telemetry list receives only transformed input. Raw fields are synthetic inspection data, not production logging guidance.

In [ ]:
from models.demo import demo_response
observed_inputs = []
baseline = {item["id"]: guarded_request(item["text"], demo_response, observed_inputs) for item in fixtures}
pprint(baseline["email"])
pprint({"telemetry_inputs": observed_inputs, "local_responder_calls": len(observed_inputs), "model_api_calls": 0})
assert "alex@example.test" not in baseline["email"]["responder_input"]
assert "hr@example.test" not in baseline["email"]["exposed"]

## Inspect a real miss in this implementation

Parenthesized phones do not match the baseline pattern, so this synthetic phone reaches the local responder. An obfuscated email is also missed. These are visible failures of the small matcher.

In [ ]:
pprint(baseline["formatted_phone"])
assert "(202) 555-0123" in baseline["formatted_phone"]["responder_input"]
pprint(mask_text(fixtures[-1]["text"]))

## Change only formatted_phone

Replay exactly the same inputs with `formatted_phone=True`. The broader pattern adds parenthesized and spaced forms. We keep the email rule and response function unchanged.

In [ ]:
candidate_inputs = []
candidate = {item["id"]: guarded_request(item["text"], demo_response, candidate_inputs, formatted_phone=True) for item in fixtures}
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline_input": baseline[name]["responder_input"],
            "candidate_input": candidate[name]["responder_input"], "exposed": candidate[name]["exposed"]})

## Check the improvement and remaining limit

The formatted phone is now masked; the obfuscated email remains. Different countries, spelling tricks, and nonpersonal number strings need separate evaluation. Explain the input and output boundary using the actual local invocation evidence.

In [ ]:
assert "(202) 555-0123" not in candidate["formatted_phone"]["responder_input"]
assert candidate["obfuscated_email"]["responder_input"] == fixtures[-1]["text"]
assert len(candidate_inputs) == len(fixtures)
assert all(result["model_api_calls"] == 0 for result in candidate.values())
pprint({"remaining_miss": candidate["obfuscated_email"]["responder_input"], "local_calls": len(candidate_inputs)})